In [ ]:
from pathlib import Path
from awpy import Demo

demos = sorted(Path("../../data/demos").glob("*.dem"))
demos

In [ ]:
dem = Demo(demos[0])
dem.parse(events=[*dem.default_events, "player_blind"])

kills, damages, rounds = dem.kills, dem.damages, dem.rounds

In [ ]:
from importlib.metadata import version
print(version("awpy"))
print(kills.columns)

from awpy import plot, data
print([n for n in dir(plot) if not n.startswith("_")])
print([n for n in dir(data) if not n.startswith("_")])

In [ ]:
import json
d = json.load(open(Path("../../backend/assets/maps/map_data.json"), encoding="utf-8"))
print(d["de_cache"])

In [ ]:
import polars as pl

kills.filter(
    (pl.col("round_num") == 6) & ((pl.col("attacker_X") - 451.0).abs() < 0.1)
).select("tick", "attacker_name", "attacker_place", "attacker_X", "attacker_Y", "victim_place", "weapon")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

assets = Path("../../backend/assets/maps")
d = json.load(open(assets / "map_data.json", encoding="utf-8"))["de_cache"]
img = Image.open(assets / "de_cache.png")

r = kills.filter((pl.col("round_num") == 6) & pl.col("attacker_X").is_not_null())
px = ((r["attacker_X"] - d["pos_x"]) / d["scale"]).to_list()
py = ((d["pos_y"] - r["attacker_Y"]) / d["scale"]).to_list()

plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.scatter(px, py, c="lime", s=60)
for x, y, place in zip(px, py, r["attacker_place"]):
    plt.annotate(place, (x, y), color="white", fontsize=9)
plt.axis("off")
plt.show()